In [1]:
import ee
ee.Authenticate()
ee.Initialize()


In [5]:
import ee

# Autenticação com escopos corretos
ee.Authenticate(scopes=[
    'https://www.googleapis.com/auth/earthengine',
    'https://www.googleapis.com/auth/drive'
])

ee.Initialize()



Successfully saved authorization token.


In [21]:
# ================================
# CONFIGURAÇÕES GERAIS
# ================================
import ee
ee.Initialize()

# MODIFICADO: Período de 2005 a 2019
start_date = '2000-01-01'
end_date = '2024-01-01'

aoi = ee.Geometry.Polygon([
    [[-77.95,-8.69],[-77.80,-10.01],[-76.95,-9.99],[-77.25,-8.62],[-77.95,-8.69]]
]);

# ================================
# VARIÁVEIS DE CONTROLE - APENAS MODIS LST ATIVO
# ================================
BAIXAR_MODIS_LST = True      # ✅ ATIVO - Apenas este
BAIXAR_MODIS_NDVI = False    # ❌ DESATIVADO  
BAIXAR_MODIS_EVI = False     # ❌ DESATIVADO
BAIXAR_GPM_PRECIP = False    # ❌ DESATIVADO
BAIXAR_CAMS_TCWV = False     # ❌ DESATIVADO
BAIXAR_MODIS_ALBEDO = False  # ❌ DESATIVADO
BAIXAR_ESA_WORLDCOVER = False # ❌ DESATIVADO

print("✅ Configurações carregadas!")
print(f"📅 Período: {start_date} a {end_date} (15 anos)")
print(f"📡 Apenas MODIS LST será baixado")
print("🌍 Área de interesse definida")

✅ Configurações carregadas!
📅 Período: 2000-01-01 a 2024-01-01 (15 anos)
📡 Apenas MODIS LST será baixado
🌍 Área de interesse definida


In [22]:
# ================================
# FUNÇÃO AUXILIAR PARA ADICIONAR LAT/LON
# ================================
def adicionar_coordenadas(image):
    """Adiciona bandas de latitude e longitude à imagem"""
    coords = ee.Image.pixelLonLat()
    return image.addBands(coords.select(['longitude', 'latitude'], ['lon', 'lat']))

# ================================
# FUNÇÃO AUXILIAR PARA EXPORTAÇÃO (MODIFICADA)
# ================================
def export_all_images(collection, name_prefix, scale):
    """Exporta todas as imagens de uma coleção para o Google Drive COM lat/lon"""
    collection_list = collection.toList(collection.size())
    size = collection.size().getInfo()
    
    print(f"Encontradas {size} imagens para {name_prefix}")
    
    for i in range(size):
        image = ee.Image(collection_list.get(i))
        
        # ADICIONAR COORDENADAS À IMAGEM
        image_with_coords = adicionar_coordenadas(image)
        
        date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo()
        description = f"{name_prefix}_{date}"
        
        task = ee.batch.Export.image.toDrive(
            image=image_with_coords.clip(aoi),
            description=description,
            folder='GEE_EXPORTS',
            fileNamePrefix=description,
            region=aoi,
            scale=scale,
            crs='EPSG:4326',  # Garantir coordenadas geográficas
            fileFormat='GeoTIFF',
            maxPixels=1e13
        )
        task.start()
        print(f"Exportando: {description} (COM lat/lon)")
    
    print(f"✓ Todos os exports de {name_prefix} iniciados!")

In [23]:
def export_single_image(image, description, scale):
    """Exporta uma única imagem para o Google Drive COM lat/lon"""
    # ADICIONAR COORDENADAS À IMAGEM
    image_with_coords = adicionar_coordenadas(image)
    
    task = ee.batch.Export.image.toDrive(
        image=image_with_coords.clip(aoi),
        description=description,
        folder="GEE_EXPORTS",
        fileNamePrefix=description,
        region=aoi,
        scale=scale,
        crs='EPSG:4326',  # Garantir coordenadas geográficas
        fileFormat='GeoTIFF',
        maxPixels=1e13
    )
    task.start()
    print(f"Exportando: {description} (COM lat/lon)")

print("Funções auxiliares carregadas!")

Funções auxiliares carregadas!


In [24]:
# ================================
# 1. MODIS LST (Temperatura da Superfície) - COM LAT/LON
# ================================
if BAIXAR_MODIS_LST:
    print("📡 Baixando dados MODIS LST (Temperatura da Superfície) COM lat/lon...")
    
    lst = ee.ImageCollection("MODIS/061/MOD11A2") \
        .filterBounds(aoi) \
        .filterDate(start_date, end_date) \
        .select('LST_Day_1km') \
        .map(lambda img: img.multiply(0.02).subtract(273.15).copyProperties(img, img.propertyNames()))
    
    export_all_images(lst, "MODIS_LST", 1000)
else:
    print("⏭️  Pulando MODIS LST")

# ================================
# 2. MODIS NDVI (Índice de Vegetação) - COM LAT/LON
# ================================
if BAIXAR_MODIS_NDVI:
    print("🌿 Baixando dados MODIS NDVI (Índice de Vegetação) COM lat/lon...")
    
    modis_veg = ee.ImageCollection("MODIS/061/MOD13Q1") \
        .filterBounds(aoi) \
        .filterDate(start_date, end_date)
    
    ndvi = modis_veg.select('NDVI') \
        .map(lambda img: img.multiply(0.0001).copyProperties(img, img.propertyNames()))
    
    export_all_images(ndvi, "MODIS_NDVI", 250)
else:
    print("⏭️  Pulando MODIS NDVI")

# ================================
# 3. MODIS EVI (Enhanced Vegetation Index) - COM LAT/LON
# ================================
if BAIXAR_MODIS_EVI:
    print("🌱 Baixando dados MODIS EVI (Enhanced Vegetation Index) COM lat/lon...")
    
    modis_veg = ee.ImageCollection("MODIS/061/MOD13Q1") \
        .filterBounds(aoi) \
        .filterDate(start_date, end_date)
    
    evi = modis_veg.select('EVI') \
        .map(lambda img: img.multiply(0.0001).copyProperties(img, img.propertyNames()))
    
    export_all_images(evi, "MODIS_EVI", 250)
else:
    print("⏭️  Pulando MODIS EVI")

# ================================
# 4. GPM Precipitação - COM LAT/LON
# ================================
if BAIXAR_GPM_PRECIP:
    print("🌧️  Baixando dados GPM (Precipitação) COM lat/lon...")
    
    gpm = ee.ImageCollection("NASA/GPM_L3/IMERG_V06") \
        .filterBounds(aoi) \
        .filterDate(start_date, end_date) \
        .select('precipitationCal')
    
    export_all_images(gpm, "GPM_Precip", 10000)
else:
    print("⏭️  Pulando GPM Precipitação")

# ================================
# 5. CAMS TCWV (Vapor d'água Total) - COM LAT/LON
# ================================
if BAIXAR_CAMS_TCWV:
    print("💨 Baixando dados CAMS TCWV (Vapor d'água Total) COM lat/lon...")
    
    tcwv = ee.ImageCollection("ECMWF/CAMS/NRT") \
        .filterBounds(aoi) \
        .filterDate(start_date, end_date) \
        .select('total_column_water_vapour')
    
    export_all_images(tcwv, "CAMS_TCWV", 30000)
else:
    print("⏭️  Pulando CAMS TCWV")

# ================================
# 6. MODIS Albedo - COM LAT/LON
# ================================
if BAIXAR_MODIS_ALBEDO:
    print("☀️ Baixando dados MODIS Albedo COM lat/lon...")
    
    albedo = ee.ImageCollection("MODIS/061/MOD43A3") \
        .filterBounds(aoi) \
        .filterDate(start_date, end_date) \
        .select('Albedo_WSA_shortwave') \
        .map(lambda img: img.multiply(0.001).copyProperties(img, img.propertyNames()))
    
    export_all_images(albedo, "MODIS_Albedo", 500)
else:
    print("⏭️  Pulando MODIS Albedo")

# ================================
# 7. ESA WorldCover (Cobertura do Solo) - COM LAT/LON
# ================================
if BAIXAR_ESA_WORLDCOVER:
    print("🗺️  Baixando dados ESA WorldCover (Cobertura do Solo) COM lat/lon...")
    
    landcover = ee.Image("ESA/WorldCover/v100/2020")
    export_single_image(landcover, "ESA_WorldCover_2020", 10)
else:
    print("⏭️  Pulando ESA WorldCover")

# ================================
# RESUMO E STATUS
# ================================
print("\n" + "="*50)
print("📊 RESUMO DOS DOWNLOADS CONFIGURADOS:")
print("="*50)
print(f"📡 MODIS LST (Temperatura): {'✅ ATIVO' if BAIXAR_MODIS_LST else '❌ DESATIVADO'}")
print(f"🌿 MODIS NDVI (Vegetação): {'✅ ATIVO' if BAIXAR_MODIS_NDVI else '❌ DESATIVADO'}")
print(f"🌱 MODIS EVI (Vegetação Aprimorado): {'✅ ATIVO' if BAIXAR_MODIS_EVI else '❌ DESATIVADO'}")
print(f"🌧️  GPM Precipitação: {'✅ ATIVO' if BAIXAR_GPM_PRECIP else '❌ DESATIVADO'}")
print(f"💨 CAMS Vapor d'água: {'✅ ATIVO' if BAIXAR_CAMS_TCWV else '❌ DESATIVADO'}")
print(f"☀️ MODIS Albedo: {'✅ ATIVO' if BAIXAR_MODIS_ALBEDO else '❌ DESATIVADO'}")
print(f"🗺️  ESA Cobertura do Solo: {'✅ ATIVO' if BAIXAR_ESA_WORLDCOVER else '❌ DESATIVADO'}")
print("\n🔹 NOVIDADE: Todos os datasets agora incluem bandas de LATITUDE e LONGITUDE!")
print("📋 Bandas disponíveis em cada arquivo:")
print("   • Banda original do dataset (LST, NDVI, etc.)")
print("   • 'lon' = Longitude em graus")
print("   • 'lat' = Latitude em graus")
print("\n💡 Para verificar o progresso dos downloads, acesse: https://code.earthengine.google.com/tasks")
print("📁 Os arquivos serão salvos na pasta 'GEE_EXPORTS' no seu Google Drive")

# ================================
# FUNÇÃO PARA VERIFICAR ESTRUTURA DOS DADOS (OPCIONAL)
# ================================
def verificar_estrutura_dados():
    """Função para verificar se as coordenadas foram adicionadas corretamente"""
    print("\n🔍 VERIFICANDO ESTRUTURA DOS DADOS:")
    print("="*40)
    
    # Exemplo com uma imagem MODIS LST
    if BAIXAR_MODIS_LST:
        lst_sample = ee.ImageCollection("MODIS/061/MOD11A2") \
            .filterBounds(aoi) \
            .filterDate('2020-01-01', '2020-01-08') \
            .select('LST_Day_1km') \
            .first()
        
        lst_with_coords = adicionar_coordenadas(lst_sample)
        
        band_names = lst_with_coords.bandNames().getInfo()
        print(f"Bandas na imagem MODIS LST: {band_names}")
        print("✅ Coordenadas adicionadas com sucesso!")
    
    print("🎯 Agora todos os seus arquivos GeoTIFF terão:")
    print("   1. Os dados originais (temperatura, NDVI, etc.)")
    print("   2. Banda 'lon' com a longitude de cada pixel")
    print("   3. Banda 'lat' com a latitude de cada pixel")

# Executar verificação (descomente se quiser testar)
# verificar_estrutura_dados()

📡 Baixando dados MODIS LST (Temperatura da Superfície) COM lat/lon...
Encontradas 1097 imagens para MODIS_LST
Exportando: MODIS_LST_2000-02-18 (COM lat/lon)
Exportando: MODIS_LST_2000-02-26 (COM lat/lon)
Exportando: MODIS_LST_2000-03-05 (COM lat/lon)
Exportando: MODIS_LST_2000-03-13 (COM lat/lon)
Exportando: MODIS_LST_2000-03-21 (COM lat/lon)
Exportando: MODIS_LST_2000-03-29 (COM lat/lon)
Exportando: MODIS_LST_2000-04-06 (COM lat/lon)
Exportando: MODIS_LST_2000-04-14 (COM lat/lon)
Exportando: MODIS_LST_2000-04-22 (COM lat/lon)
Exportando: MODIS_LST_2000-04-30 (COM lat/lon)
Exportando: MODIS_LST_2000-05-08 (COM lat/lon)
Exportando: MODIS_LST_2000-05-16 (COM lat/lon)
Exportando: MODIS_LST_2000-05-24 (COM lat/lon)
Exportando: MODIS_LST_2000-06-01 (COM lat/lon)
Exportando: MODIS_LST_2000-06-09 (COM lat/lon)
Exportando: MODIS_LST_2000-06-17 (COM lat/lon)
Exportando: MODIS_LST_2000-06-25 (COM lat/lon)
Exportando: MODIS_LST_2000-07-03 (COM lat/lon)
Exportando: MODIS_LST_2000-07-11 (COM lat/lo